# Fungi and the Networks of Decomposition and Exchange Workflow

This notebook scaffold supports the article **Fungi and the Networks of Decomposition and Exchange**. It can be expanded with decomposition kinetics, environmental modifiers, fungal guild scenarios, biomass recovery, mycelial network efficiency, restoration-priority screening, and provenance documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
sites = pd.read_csv(article_dir / 'data' / 'decomposition_sites.csv')
sites.head()

In [ ]:
def temp_multiplier(temp, tref=10.0, q10=2.0):
    return q10 ** ((temp - tref) / 10.0)

def moisture_multiplier(moisture, m_opt=0.6, sigma=0.22):
    return np.exp(-((moisture - m_opt) ** 2) / (2 * sigma ** 2))

def quality_multiplier(lignin_n, slope=0.03):
    return np.exp(-slope * lignin_n)

def guild_multiplier(guild):
    return {'white_rot': 1.20, 'brown_rot': 0.95, 'mixed_saprotroph': 1.00, 'disturbance_simplified': 0.72}.get(guild, 1.0)

sites['k_eff'] = sites.apply(
    lambda r: r['k0'] * temp_multiplier(r['temp']) * moisture_multiplier(r['moisture']) * quality_multiplier(r['lignin_n']) * guild_multiplier(r['guild']),
    axis=1
)
sites['remaining_mass_t24'] = sites['M0'] * np.exp(-sites['k_eff'] * 24)
sites[['site', 'k_eff', 'remaining_mass_t24']].round(3)

In [ ]:
priority = pd.read_csv(article_dir / 'data' / 'restoration_priority_sites.csv')
priority['recovery_score'] = (
    0.30 * priority['mycorrhizal_inoculum'] +
    0.25 * priority['saprotroph_activity'] +
    0.20 * priority['soil_connectivity'] +
    0.15 * (1 - priority['pathogen_pressure']) +
    0.10 * (1 - priority['drought_stress'])
)
priority.sort_values('recovery_score', ascending=False).round(3)